# 🔍 Day 2: The Front Office
### AP CSA · Topic 4.14 — Linear Search Algorithms

Second period. The bell hasn't even rung yet and the front office already texted you: *"Quick — is a student with ID 40881 checked in yet? Someone's asking."*

Yesterday's grid problems all had a seating chart to lean on — rows and pods gave you structure for free. Today's data is meaner: just a **flat, unsorted list**. Names came in whatever order enrollment happened to process them. IDs aren't sequential. Scores aren't sorted. There is no shortcut hiding in the structure, because there *is* no structure.

So how do you find anything in a list with no order to exploit? **You check every element, one at a time, in order, until you find it or you run out of list.** That's it. That's linear search. It is, in a real sense, the most *honest* algorithm you'll learn all year — no shortcuts, no guessing, no assumptions about the data. Just discipline.

### The idea to hold onto for this whole notebook

**Linear search works on *any* list, sorted or not — and that generality is exactly why it can't be fast.**

Every trick you'll learn later this unit (binary search, next lesson) buys speed by *demanding* something in return: the data has to be sorted first. Linear search asks for nothing and promises nothing back except "I will eventually tell you the truth, checking one item at a time." Keep that trade-off in your head — it's the single idea that makes the *next* lesson make sense before you've even started it.

**By the end of this notebook you will be able to:**
- Implement linear search on an array, returning the index of a match or a clear "not found" signal
- Correctly compare primitives (`==`) versus objects like `String` (`.equals()`) while searching — and know exactly why mixing them up produces bugs that *look* like they should work
- Search based on a **condition**, not just exact equality (find the *first* element that satisfies some rule)
- Correctly search a **partially filled array** — respecting a logical "how many slots are actually in use" count instead of blindly trusting `.length`
- Collect **every** match instead of stopping at the first one
- Search an `ArrayList` and explain how that differs syntactically (not conceptually) from searching an array
- Reason about linear search's best case, worst case, and why "not found" is *always* the worst case


## The Roster

Twelve students. Names, IDs, and most recent quiz scores — three parallel arrays, same index means the same student in all three. Notice: **nothing here is sorted.** That's not an oversight, it's the entire point of this lesson.


In [ ]:
String[] names       = {"Maya Chen", "Diego Ruiz", "Aaliyah Brooks", "Sam Okafor", "Jordan Blake",
                         "Priya Nair", "Liam Carter", "Zoe Fischer", "Noah Kim", "Ines Delgado",
                         "Marcus Webb", "Grace Liu"};
int[] studentIDs     = {40217, 40593, 40108, 40772, 40364, 40881, 40255, 40940, 40033, 40108, 40699, 40128};
int[] quizScores     = {87, 54, 92, 71, 88, 46, 79, 95, 62, 83, 58, 90};

System.out.println("Roster loaded: " + names.length + " students.");


## Chapter 1 — No Shortcuts Allowed

Picture a hallway of lockers with no numbers on them, assigned in whatever order students showed up on the first day. You're looking for locker `40881`. There is exactly one strategy that works: **start at locker 0, check it, and if it's not the one, move to locker 1. Repeat until you find it or you've checked every locker in the hallway.**

You cannot skip ahead. You cannot guess "it's probably in the back half." The data gives you *zero* information about where anything is relative to anything else — so the algorithm has to earn every answer the hard way, one comparison at a time. That's what makes it "linear": the number of checks grows in direct, straight-line proportion to how far into the list the answer happens to be.

```java
for (int i = 0; i < arr.length; i++) {
    if (arr[i] == target) {
        return i;          // found it — stop immediately, no reason to keep checking
    }
}
return -1;                 // walked the entire list, target isn't here
```

Two design decisions are baked into that skeleton, and both are constantly graded on the AP exam:
- **`return i` the instant you find a match.** Don't set a flag and keep looping "just to be safe" — that wastes work and, worse, if you're tracking "first match" and don't stop, a careless implementation can silently overwrite your answer with a *later* match.
- **`return -1` only after the loop completes naturally.** `-1` isn't a real index, so it's a safe, unambiguous signal that means "not present" — same sentinel idea as `-1` for an empty seat in yesterday's grid. Sentinel values keep showing up because "how do I signal *nothing*" is a problem that never goes away in programming.


## Chapter 2 — The Basic Search

**Your task:** write `indexOfID`, which linearly searches `studentIDs` for `targetID` and returns its index, or `-1` if it isn't in the roster.


In [ ]:
public static int indexOfID(int[] ids, int targetID) {
    for (int i = 0; i < /* TODO */; i++) {
        if (/* TODO */) {
            return /* TODO */;
        }
    }
    return /* TODO */;
}

System.out.println(indexOfID(studentIDs, 40881));  // expect a real index
System.out.println(indexOfID(studentIDs, 99999));  // expect -1


<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int indexOfID(int[] ids, int targetID) {
    for (int i = 0; i < ids.length; i++) {
        if (ids[i] == targetID) {
            return i;
        }
    }
    return -1;
}
```

`indexOfID(studentIDs, 40881)` should return `5` — Priya Nair. `indexOfID(studentIDs, 99999)` should return `-1`, since no student has that ID; the loop runs all 12 comparisons and falls through to the final `return -1` untouched.

`==` is correct here because `int` is a primitive — comparing two `int`s with `==` compares their actual values directly. That's about to matter a lot in the next chapter, because it stops being true the moment you're searching for something that *isn't* a primitive.
</details>


## Chapter 3 — Best Case, Worst Case, and Why "Not Found" Is Always the Worst Case

Add a comparison counter to `indexOfID` and run it against a few different targets. Watch the count change.

**Your task:** write `indexOfIDCounted`, identical to `indexOfID` but it also prints how many comparisons it made before returning.


In [ ]:
public static int indexOfIDCounted(int[] ids, int targetID) {
    int comparisons = 0;
    for (int i = 0; i < ids.length; i++) {
        comparisons++;   // TODO: where exactly should this line live relative to the comparison itself? think before you run it.
        if (ids[i] == targetID) {
            System.out.println("Found after " + comparisons + " comparison(s).");
            return i;
        }
    }
    System.out.println("Not found after " + comparisons + " comparison(s).");
    return -1;
}

indexOfIDCounted(studentIDs, 40217);   // first element — best case
indexOfIDCounted(studentIDs, 40128);   // last element — worst case (found)
indexOfIDCounted(studentIDs, 99999);   // absent — worst case (not found)


<details>
<summary>✅ What you should see, and why it matters</summary>

- `40217` (index 0): **1 comparison.** Best case — the target happens to be the very first thing checked. This is **O(1)**: constant, independent of how big the list is.
- `40128` (index 11, the last slot): **12 comparisons.** Worst case *among found targets* — every single element had to be checked before reaching the match.
- `99999` (not present at all): **12 comparisons.** Also worst case, and for a subtle reason worth sitting with: **the algorithm has no way to know a target is missing without checking literally everything first.** There's no early clue that says "give up now." This is why "not found" is *always* tied for the worst-case scenario, no matter what you're searching for — you can't prove absence without exhausting the list.

This is **O(n)** worst-case behavior: double the size of the roster, and in the worst case, you double the number of comparisons. That scaling relationship — not the exact comparison count — is the actual thing "Big-O" is describing, and it's the fact that will directly motivate why binary search (next lesson, once your data is sorted) can beat this by throwing away *half* the remaining list with every single check instead of one element at a time.
</details>

> ### 🔧 Java Hack — don't let an early `return` fool you about "how much work" you did
> A method that returns after 1 comparison and one that returns after 12 both *look* instant when you run them — Java is fast enough that you'll never perceive the difference by eye. Big-O reasoning exists precisely because "did it feel slow" is useless information at this scale; you have to reason about the *shape* of the algorithm, not how it felt to run once on 12 elements. On a roster of 12,000, that same worst case is 12,000 comparisons, and now you'd feel it.


## Chapter 4 — Searching for a Name (the `.equals()` trap)

The front office's next question: *"Is 'Marcus Webb' on your roster?"* Now you're searching `names`, a `String[]` — and `String` is not a primitive. It's an object. Comparing objects with `==` doesn't ask "do these hold the same value," it asks **"are these the literal same object in memory."** For two separately-constructed `String`s that happen to contain identical text, `==` can betray you.

**Your task:** write `indexOfName`, searching `names` for `targetName` using the **correct** comparison for objects.


In [ ]:
public static int indexOfName(String[] roster, String targetName) {
    for (int i = 0; i < roster.length; i++) {
        if (/* TODO: compare roster[i] and targetName the correct way for objects */) {
            return i;
        }
    }
    return -1;
}

System.out.println(indexOfName(names, "Marcus Webb"));
System.out.println(indexOfName(names, "Casey Nguyen"));  // not on the roster


<details>
<summary>💡 Hint</summary>

The method you want is a method *on* the `String` itself: `roster[i].equals(targetName)`. It compares actual character content, not memory identity.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int indexOfName(String[] roster, String targetName) {
    for (int i = 0; i < roster.length; i++) {
        if (roster[i].equals(targetName)) {
            return i;
        }
    }
    return -1;
}
```

`indexOfName(names, "Marcus Webb")` returns `10`. `indexOfName(names, "Casey Nguyen")` returns `-1`.

Here's the trap in full: `roster[i] == targetName` will often *appear* to work in quick tests, because Java caches certain `String` literals in a shared pool, so two identical literals can accidentally be the same object. But the instant one of those `String`s is built at runtime — read from user input, built with `+`, returned from a method — that coincidence disappears and `==` starts silently returning `false` for strings that are, for every human purpose, identical. **The rule that never fails you:** primitives (`int`, `double`, `boolean`, `char`) compare with `==`. Objects (`String`, and anything else with a capital-letter type) compare with `.equals()`. Memorize the rule, not the coincidence.
</details>


## Chapter 5 — Search Doesn't Mean "Equals"

So far every search has hunted for one exact value. But the front office's real questions are rarely that clean: *"Who's the first student in the roster with a failing quiz score?"* There's no single target value to match against — you're searching for the first element that satisfies a **condition**.

The algorithm's shape doesn't change at all. Only the `if` changes — from `arr[i] == target` to whatever boolean expression describes what you're actually looking for.

**Your task:** write `indexOfFirstBelow`, returning the index of the first student in `quizScores` with a score strictly below `threshold`, or `-1` if nobody qualifies.


In [ ]:
public static int indexOfFirstBelow(int[] scores, int threshold) {
    for (int i = 0; i < scores.length; i++) {
        if (/* TODO */) {
            return i;
        }
    }
    return -1;
}

int idx = indexOfFirstBelow(quizScores, 60);
System.out.println(names[idx] + " scored " + quizScores[idx]);


<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int indexOfFirstBelow(int[] scores, int threshold) {
    for (int i = 0; i < scores.length; i++) {
        if (scores[i] < threshold) {
            return i;
        }
    }
    return -1;
}
```

With `threshold = 60`, this returns index `1` — **Diego Ruiz, score 54** — the *first* student below 60, even though Priya Nair (index 5, score 46) has a *lower* score. "First to satisfy the condition" and "most extreme value" are genuinely different questions — the second one is the "track the champion while you walk" pattern from yesterday's capstone, not a linear search at all. Confusing "find the first match" with "find the best match" is a very common way to answer the *wrong question correctly* on an FRQ — the code compiles, it returns a real index, and it's still not what was asked.
</details>


## Chapter 6 — The Sign-In Sheet (searching a *partially filled* array)

Here's a nuance that quietly wrecks otherwise-correct search code: **an array's `.length` is its *capacity*, not necessarily how much of it is actually in use.**

The office set up a sign-in sheet for an assembly with room for 15 names, but as of right now, only 8 students have actually checked in. The other 7 slots are just sitting there holding Java's default `int` value, `0` — not real data, just unused space:

```java
int[] signInSheet = new int[15];   // capacity: 15
int numSignedIn = 8;               // actual, meaningful entries: only the first 8
```

If you search `signInSheet` using `signInSheet.length` as your bound, you are searching **7 slots of pure garbage** right alongside your real data — and if you were ever searching for a target value of `0`, you'd get a wildly wrong "found it!" pointing at an empty slot that was never really data. The fix is simple but easy to forget under pressure: **loop bound to the logical size (`numSignedIn`), never to `.length`, whenever the two aren't guaranteed to be the same.**

**Your task:** write `isSignedIn`, checking whether `targetID` appears among the *actual* sign-ins — not the unused capacity.


In [ ]:
int[] signInSheet = new int[15];
int numSignedIn = 8;
signInSheet[0] = 40217;
signInSheet[1] = 40593;
signInSheet[2] = 40108;
signInSheet[3] = 40772;
signInSheet[4] = 40364;
signInSheet[5] = 40881;
signInSheet[6] = 40255;
signInSheet[7] = 40940;
// signInSheet[8] through [14] are still just 0 — nobody's signed in there yet.

public static boolean isSignedIn(int[] sheet, int numSignedIn, int targetID) {
    for (int i = 0; i < /* TODO: the CORRECT bound — think carefully about which variable belongs here */; i++) {
        if (sheet[i] == targetID) {
            return true;
        }
    }
    return false;
}

System.out.println(isSignedIn(signInSheet, numSignedIn, 40940));  // expect true  (really signed in)
System.out.println(isSignedIn(signInSheet, numSignedIn, 0));       // expect false (0 is just unused space, not a real ID)


<details>
<summary>💡 Hint</summary>

The bound is `numSignedIn`, not `sheet.length`. Prove to yourself why: swap it to `sheet.length` and re-run the second test case. What does it return, and *why* is that answer actually wrong even though the code "works"?
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static boolean isSignedIn(int[] sheet, int numSignedIn, int targetID) {
    for (int i = 0; i < numSignedIn; i++) {
        if (sheet[i] == targetID) {
            return true;
        }
    }
    return false;
}
```

If you accidentally loop to `sheet.length` (15) instead of `numSignedIn` (8), searching for `0` returns `true` — because slot 8 genuinely does contain the value `0`, Java's default filler. The code isn't buggy in the sense of crashing or having a typo; it's buggy because it doesn't understand the difference between *the array's capacity* and *what the array is currently modeling*. **This exact array-vs-size distinction shows up constantly on the real exam** — often stated explicitly in a prompt as something like "the array `arr` has `size` meaningful elements, though `arr.length` may be larger" — and it's a near-guaranteed lost point for anyone who's never been burned by it before today.
</details>


## Chapter 7 — When You Don't Stop at the First One

Not every search wants "the first match, then quit." The front office now wants a full list: *"Give me every student currently failing."* Stopping after Diego Ruiz (the first one, from Chapter 5) would be actively wrong here — you need **all** of them, which means the early `return` that made every previous method efficient is exactly the thing you must *not* do this time.

**Your task:** write `indexesBelow`, returning an `ArrayList<Integer>` containing the index of *every* student scoring below `threshold`, in order.


In [ ]:
import java.util.ArrayList;

public static ArrayList<Integer> indexesBelow(int[] scores, int threshold) {
    ArrayList<Integer> matches = new ArrayList<>();
    for (int i = 0; i < scores.length; i++) {
        if (scores[i] < threshold) {
            /* TODO: record this index — don't return, there might be more coming */
        }
    }
    return matches;
}

ArrayList<Integer> failing = indexesBelow(quizScores, 60);
for (int i : failing) {
    System.out.println(names[i] + ": " + quizScores[i]);
}


<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static ArrayList<Integer> indexesBelow(int[] scores, int threshold) {
    ArrayList<Integer> matches = new ArrayList<>();
    for (int i = 0; i < scores.length; i++) {
        if (scores[i] < threshold) {
            matches.add(i);
        }
    }
    return matches;
}
```

Expect three names back: **Diego Ruiz (54), Priya Nair (46), Marcus Webb (58).** Notice this method's loop **never contains an early `return` inside the `if`.** That's not an oversight — it's the entire structural difference between "find the first/any match" (Chapters 2–6) and "find every match" (this chapter). Recognizing which of these two shapes a question is actually asking for, *before* you write a single line, is exam-critical: adding a premature `return` here would silently produce a list of at most one item, and the code would still compile and run without complaint.
</details>


## Chapter 8 — Same Algorithm, Different Syntax (searching an `ArrayList`)

The waitlist for this class isn't a fixed-size array at all — students add themselves and drop off all the time, so it lives in an `ArrayList<String>`. **The linear search *algorithm* doesn't change one bit.** Only the syntax for "how big is this thing" and "get me the element at position i" changes.

| | Array | ArrayList |
|---|---|---|
| Size | `arr.length` (no parentheses — it's a field) | `list.size()` (a method — parentheses required) |
| Get element `i` | `arr[i]` | `list.get(i)` |
| Compare objects | `.equals()` | `.equals()` — no change |

**Your task:** write `indexInWaitlist`, searching the `ArrayList<String>` for `targetName`.


In [ ]:
import java.util.ArrayList;
import java.util.Arrays;

ArrayList<String> waitlist = new ArrayList<>(Arrays.asList("Owen Price", "Talia Wong", "Devon Marsh", "Priya Sharma"));

public static int indexInWaitlist(ArrayList<String> list, String targetName) {
    for (int i = 0; i < /* TODO */; i++) {
        if (/* TODO */) {
            return i;
        }
    }
    return -1;
}

System.out.println(indexInWaitlist(waitlist, "Devon Marsh"));
System.out.println(indexInWaitlist(waitlist, "Casey Nguyen"));


<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int indexInWaitlist(ArrayList<String> list, String targetName) {
    for (int i = 0; i < list.size(); i++) {
        if (list.get(i).equals(targetName)) {
            return i;
        }
    }
    return -1;
}
```

`indexInWaitlist(waitlist, "Devon Marsh")` returns `2`. `indexInWaitlist(waitlist, "Casey Nguyen")` returns `-1`. Put this side by side with `indexOfName` from Chapter 4 — the *logic* is character-for-character identical: same loop shape, same early return, same `.equals()` for object comparison, same `-1` sentinel. Only `roster.length` became `list.size()` and `roster[i]` became `list.get(i)`. **This is the actual lesson of this chapter:** once you understand linear search as an algorithm and not as "a specific block of Java syntax," you can port it to any linear collection in about four keystrokes.

> ### 🔧 Java Hack — `ArrayList` has no `for-each` shortcut for index-based needs
> `for (String s : waitlist)` works great if you only need each *value*. The moment you need to `return i` or report a position, you're back to an indexed loop with `list.get(i)` — same trade-off as enhanced-`for` on arrays from yesterday's Chapter 4. Values-only versus values-with-position is a decision you make on *every* collection type, not just arrays.
</details>


## Chapter 9 — Find the Bug

A previous sub wrote this method to check if *any* student is failing (`score < 60`). It's supposed to return `true` or `false`. It always returns `false`, even though you know from Chapter 7 that three students are failing.

```java
public static boolean anyFailing(int[] scores) {
    boolean failing = false;
    for (int i = 0; i < scores.length; i++) {
        if (scores[i] < 60) {
            failing = true;
        }
        return failing;
    }
    return failing;
}
```

Find the bug **by tracing it, not by guessing.** Walk through `scores = {87, 54, ...}` by hand, one iteration of the loop at a time, and watch exactly when the method actually returns.


In [ ]:
public static boolean anyFailing(int[] scores) {
    boolean failing = false;
    for (int i = 0; i < scores.length; i++) {
        if (scores[i] < 60) {
            failing = true;
        }
        return failing;   // TODO: this line is in the wrong place. fix it.
    }
    return failing;
}

System.out.println(anyFailing(quizScores));  // expect true


<details>
<summary>💡 Hint</summary>

`return failing;` sits **inside the `for` loop but outside the `if`** — so it runs after the very first iteration, no matter what. On iteration `i = 0`, `scores[0]` is `87`, which is not `< 60`, so `failing` stays `false` — and then the method returns that `false` immediately, before ever looking at index 1. The loop never gets a second iteration.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static boolean anyFailing(int[] scores) {
    for (int i = 0; i < scores.length; i++) {
        if (scores[i] < 60) {
            return true;   // found one — we can answer immediately
        }
    }
    return false;   // checked everything, nobody qualified
}
```

The cleanest fix drops the `failing` variable entirely: the instant you find *any* qualifying element, the answer to "does any exist?" is settled — `return true` right then. If the loop finishes without ever hitting that line, the answer is `false`. This is the exact same "does the target exist" question as `indexOfID`, just reporting a `boolean` instead of a location — same family of algorithm, smaller return type. Compare this bug to Chapter 7's "don't return early" warning: the *skill* being tested in both places is the same one — knowing precisely which line your `return` belongs on, because moving it one brace up or down completely changes what the method computes, with zero compiler complaints either way.
</details>


## Capstone — The Duplicate ID Detector

IT just flagged a possible data-entry error: two students on the roster might share the same student ID. Your job: find out for sure, and report exactly who.

There's no single "target" to search for here — you need to compare **every student's ID against every other student's ID** and report any pair that matches. That means a linear search *nested inside* a loop over every starting position: for each student `i`, linearly search the *rest* of the roster (`j` from `i + 1` onward, so you never re-check a pair backwards or compare a student against themselves) for a matching ID.

**Your task:** write `findDuplicateIDs`, which prints every duplicate pair it finds, in the format `"Duplicate ID <id> found at indices <i> and <j> (<name i> & <name j>)"`.


In [ ]:
public static void findDuplicateIDs(int[] ids, String[] roster) {
    for (int i = 0; i < ids.length; i++) {
        for (int j = /* TODO: where should this inner search start, and why? */; j < ids.length; j++) {
            if (/* TODO */) {
                System.out.println("Duplicate ID " + ids[i] + " found at indices " + i + " and " + j
                        + " (" + roster[i] + " & " + roster[j] + ")");
            }
        }
    }
}

findDuplicateIDs(studentIDs, names);


<details>
<summary>💡 Hint — why `j` starts at `i + 1`</summary>

If `j` started at `0`, you'd compare every student against *themselves* (`i == j`, always "matching," always useless), and you'd report every real duplicate pair **twice** — once as `(i, j)` and again later as `(j, i)`. Starting `j` at `i + 1` guarantees each pair is checked exactly once, and only ever compares *different* students.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static void findDuplicateIDs(int[] ids, String[] roster) {
    for (int i = 0; i < ids.length; i++) {
        for (int j = i + 1; j < ids.length; j++) {
            if (ids[i] == ids[j]) {
                System.out.println("Duplicate ID " + ids[i] + " found at indices " + i + " and " + j
                        + " (" + roster[i] + " & " + roster[j] + ")");
            }
        }
    }
}
```

This should surface exactly one pair: **ID 40108, indices 2 and 9 — Aaliyah Brooks & Ines Delgado.**

Sit with the cost of this for a second, because it's the real payoff of the whole notebook: for every one of the 12 students, you're running something close to a full linear search across the rest of the roster. That's roughly `12 × 12` comparisons in the worst case — **quadratic**, not linear, even though every individual inner search is nothing but ordinary linear search. This is exactly how algorithms genuinely get slow in practice: not by using a bad technique, but by nesting a perfectly good O(n) technique inside another loop without noticing the multiplication. Recognizing "a search inside a loop over the whole list" as a red flag for O(n²) is a real, transferable skill — and it's precisely the kind of awareness that separates competent code from code a 5-scorer would flag and think twice about.
</details>


## 🎯 AP Exam-Ready Checkpoint

- [ ] **Correct comparison operator** — `==` for primitives, `.equals()` for objects (`String`s especially — this is one of the single most common silent-bug sources on the entire exam)
- [ ] **Early exit exactly when it should be there, and only then** — `return` the moment you find what you need *if* the task wants the first/any match; **no** early exit if the task wants every match
- [ ] **Correct sentinel for "not found"** — typically `-1` for an index, `false` for a yes/no question — and that `return` sits *after* the loop, reached only when the loop finishes without ever finding a match
- [ ] **Correct loop bound when size ≠ capacity** — a logical "how many are actually here" variable, never a blind `.length`/`.size()` when the two might differ
- [ ] **Method signature matches exactly what's asked** — return type, parameter types and order, and (on the real exam) the exact given method name

### One more rep — try this cold

> Write a method `firstDoubleFailure(int[] scores)` that returns the index of the **first** position `i` (where `i` is at least 1) such that **both** `scores[i]` and `scores[i - 1]` are below 60 — i.e., the first place two failing scores happen to sit back-to-back in the roster. Return `-1` if no such position exists.
>
> *(Hint: on `quizScores`, no two failing scores are actually adjacent to each other — so the correct, fully-reasoned answer for this exact roster is `-1`. Getting `-1` doesn't mean your code is broken; it means you need a test case where it should find something, too, to be sure.)*


In [ ]:
public static int firstDoubleFailure(int[] scores) {
    // TODO: it's yours. Think about the loop's starting index carefully —
    // scores[i - 1] needs to be a valid, in-bounds access on every iteration.
    return -1;
}

System.out.println(firstDoubleFailure(quizScores));  // expect -1 on this roster

int[] testCase = {90, 55, 40, 88};   // indices 1 and 2 are both failing, back-to-back
System.out.println(firstDoubleFailure(testCase));    // expect 2


<details>
<summary>✅ Model solution</summary>

```java
public static int firstDoubleFailure(int[] scores) {
    for (int i = 1; i < scores.length; i++) {
        if (scores[i] < 60 && scores[i - 1] < 60) {
            return i;
        }
    }
    return -1;
}
```

The loop **starts at `i = 1`, not `0`** — that's the whole puzzle of this problem. Starting at `0` would make `scores[i - 1]` evaluate to `scores[-1]` on the very first iteration, an instant `ArrayIndexOutOfBoundsException`. Whenever a search compares an element to its *neighbor*, always ask which end of the array that neighbor-check could walk off of, and start (or end) your bound one step in from that edge.
</details>


## 🔁 Before you close this notebook

1. Every method in this notebook shares one skeleton: a loop, a condition, and a decision about *when* to `return`. What's the one-sentence rule that tells you whether a problem wants an early exit (Chapters 2, 4, 5, 6, 9) or wants you to keep going and collect everything (Chapter 7)?
2. You searched an `int[]`, a `String[]`, a partially-filled `int[]`, and an `ArrayList<String>` — four different containers, one algorithm. Name the two things that actually changed between them, and the one thing that never did.
3. The Duplicate ID Detector nested a linear search inside a loop and quietly became O(n²). Where else in your life-as-code (or in yesterday's grid notebook) have you already written something that was secretly quadratic without meaning to?

**The setup for next time, stated plainly:** linear search's entire personality — check everything, in order, no assumptions — exists because it refuses to assume the data is sorted. The instant you're *guaranteed* sorted data, a smarter algorithm gets to cheat: it can look at the middle element, decide the whole left or right half is impossible, and throw half the remaining list away without checking a single element in it. That algorithm is next. You already understand why it's necessary — you just spent an entire notebook doing it the honest, expensive way.
